# Демо проектных упражнений модуля 16 — Обучаемая VLM и маршрутизация её ответов

**Демо-реализация, не готовое решение полномасштабного проекта.** Изображения двух столбиков кодируются непосредственно из пикселей. Обучаем две маленькие сети с разным числом позиций, проверяем генерацию и пустое изображение. Затем превращаем сохранённые ответы в разрешённые отметки на учебной схеме. Внешних API и действий нет.
Все входы малы, детерминированы и работают без сети на CPU. Цель — проверить
интерфейсы, метрики, инварианты и структуру отчёта до расходования GPU/API-бюджета.

In [ ]:
from __future__ import annotations
import importlib.util, subprocess, sys
for package in ("numpy", "matplotlib"):
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])


import math
import random
import time
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import numpy as np

SEED = 20260801
random.seed(SEED)
np.random.seed(SEED)
print(f"воспроизводимый тестовый запуск; seed={SEED}")
DEMO_LANGUAGE = 'ru'


Проверьте, как создаются данные: все 16 пар высот встречаются в обучении при яркости 0.6 и 1.0, а при оценке — при яркости 0.8. Сети получают пиксели и постоянный запрос, а не численные высоты. Обе обучаются 240 шагов; режим с четырьмя позициями сначала объединяет соседние патчи. На каждом шаге запрос предсказывает ответ, а ответ — EOS. Сопоставьте этот сдвиг с позициями функции потерь.

In [ ]:
import importlib.util, subprocess, sys, json
if importlib.util.find_spec("torch") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch"])
"""A small, fully trained pixels-to-token model; no pretrained VLM or external data.

Images are generated by this module. Held-out images change rendering brightness,
not the underlying bar-height combinations. This tests rendering transfer only.
"""
import io
import math
import torch
from torch import nn
from torch.nn import functional as F


def bar_data():
    """Return disjoint 8x8 grayscale renders of two bars with heights 1..4."""
    images, labels, split = [], [], []
    for brightness in (0.6, 1.0, 0.8):
        for left in range(1, 5):
            for right in range(1, 5):
                image = torch.zeros(1, 8, 8)
                image[0, 7-left:7, 1:3] = brightness
                image[0, 7-right:7, 5:7] = brightness
                images.append(image)
                # IDs: prompt=0, left=1, equal=2, right=3, EOS=4.
                labels.append([1 if left > right else 3 if right > left else 2, 4])
                split.append(brightness == 0.8)
    x, y, test = torch.stack(images), torch.tensor(labels), torch.tensor(split)
    assert len({row.numpy().tobytes() for row in x}) == 48
    return x[~test], y[~test], x[test], y[test]


class TinyVLM(nn.Module):
    """Patch encoder -> MLP connector -> one causal decoder -> answer and EOS.

    merge=2 uses lossless pixel unshuffle BEFORE the learned encoder: it changes
    the number of positions, not the number of observed pixels. This is not a ViT.
    """
    def __init__(self, merge=1):
        super().__init__()
        if type(merge) is not int or merge not in (1, 2):
            raise ValueError("merge must be 1 or 2")
        self.merge = merge
        self.n_visual = 16 // merge**2
        self.encoder = nn.Linear(4 * merge**2, 16)
        self.projector = nn.Sequential(nn.Linear(16, 32), nn.GELU(), nn.Linear(32, 32))
        self.embedding = nn.Embedding(5, 32)
        self.position = nn.Parameter(torch.randn(1, self.n_visual + 3, 32) * 0.02)
        self.norm1 = nn.LayerNorm(32)
        self.qkv = nn.Linear(32, 96)
        self.out = nn.Linear(32, 32)
        self.norm2 = nn.LayerNorm(32)
        self.ff = nn.Sequential(nn.Linear(32, 64), nn.GELU(), nn.Linear(64, 32))
        self.head = nn.Linear(32, 5)

    def forward(self, images, text):
        # unfold: [B,C,H,W] -> [B,16,4], row-major 2x2 patches.
        b = len(images)
        patches = F.unfold(images, kernel_size=2, stride=2).transpose(1, 2)
        if self.merge == 2:
            patches = patches.reshape(b, 2, 2, 2, 2, 4).permute(0, 1, 3, 2, 4, 5).reshape(b, 4, 16)
        visual = self.projector(torch.tanh(self.encoder(patches)))
        x = torch.cat([visual, self.embedding(text)], dim=1)
        x = x + self.position[:, :x.shape[1]]
        q, k, v = self.qkv(self.norm1(x)).chunk(3, dim=-1)
        scores = q @ k.transpose(-1, -2) / math.sqrt(32)
        allowed = torch.ones(x.shape[1], x.shape[1], dtype=torch.bool, device=x.device).tril()
        attention = scores.masked_fill(~allowed, float('-inf')).softmax(-1)
        x = x + self.out(attention @ v)
        x = x + self.ff(self.norm2(x))
        return self.head(x)

    def loss(self, images, answers):
        # Prompt predicts answer; answer predicts EOS. Exactly one causal shift.
        text = torch.cat([torch.zeros(len(images), 1, dtype=torch.long), answers[:, :1]], dim=1)
        logits = self(images, text)[:, self.n_visual:]
        return F.cross_entropy(logits.reshape(-1, 5), answers.reshape(-1))

    @torch.no_grad()
    def generate(self, images):
        text = torch.zeros(len(images), 1, dtype=torch.long)
        for _ in range(2):
            token = self(images, text)[:, -1].argmax(-1, keepdim=True)
            text = torch.cat([text, token], dim=1)
        return text[:, 1:]


def run_vision_lab(steps=240):
    """Train two declared models, save/reload, and return stored predictions."""
    if type(steps) is not int or steps <= 0:
        raise ValueError("steps must be a positive integer")
    torch.set_num_threads(1)
    torch.use_deterministic_algorithms(True)
    train_x, train_y, test_x, test_y = bar_data()
    metrics, records = {}, {}
    for merge in (1, 2):
        torch.manual_seed(1600 + merge)
        model = TinyVLM(merge)
        initial = float(model.loss(test_x, test_y).detach())
        optimizer = torch.optim.AdamW(model.parameters(), lr=0.005, weight_decay=0.01)
        losses = []
        for _ in range(steps):
            loss = model.loss(train_x, train_y)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in model.parameters())
            optimizer.step()
            losses.append(float(loss.detach()))
        model.eval()
        predictions = model.generate(test_x)
        blank_predictions = model.generate(torch.zeros_like(test_x))
        train_predictions = model.generate(train_x)
        # The first answer logit cannot see the teacher-forced answer token.
        a = torch.zeros(len(test_x), 2, dtype=torch.long)
        b = a.clone(); b[:, 1] = 3
        with torch.no_grad():
            torch.testing.assert_close(model(test_x, a)[:, model.n_visual], model(test_x, b)[:, model.n_visual])
        # Serialization must preserve the model's actual outputs.
        buffer = io.BytesIO()
        torch.save(model.state_dict(), buffer)
        buffer.seek(0)
        restored = TinyVLM(merge)
        restored.load_state_dict(torch.load(buffer, weights_only=True))
        restored.eval()
        assert torch.equal(restored.generate(test_x), predictions)
        # Freezing the decoder must NOT cut the input gradient to the projector.
        for name, p in model.named_parameters():
            p.requires_grad_(name.startswith('projector.'))
            p.grad = None
        model.loss(train_x, train_y).backward()
        assert sum(float(p.grad.abs().sum()) for p in model.projector.parameters()) > 0
        assert all(p.grad is None for name, p in model.named_parameters() if not name.startswith('projector.'))
        key = str(merge)
        metrics[key] = dict(visual_tokens=model.n_visual, parameters=sum(p.numel() for p in model.parameters()),
            train_images=len(train_x), test_images=len(test_x), steps=steps,
            initial_test_ce=initial, final_test_ce=float(model.loss(test_x, test_y).detach()),
            train_correct=int((train_predictions == train_y).all(-1).sum()),
            exact_correct=int((predictions == test_y).all(-1).sum()),
            blank_correct=int((blank_predictions == test_y).all(-1).sum()))
        records[key] = dict(predictions=predictions.tolist(), targets=test_y.tolist(),
                            blank_predictions=blank_predictions.tolist(), loss=losses)
    return metrics, records



plt.rcParams.update({"font.family": "DejaVu Sans", "figure.figsize": (11, 6),
    "figure.dpi": 150, "font.size": 12, "axes.titlesize": 17, "axes.labelsize": 13,
    "xtick.labelsize": 11, "ytick.labelsize": 11, "legend.fontsize": 11})
metrics, records = run_vision_lab()
print(json.dumps(metrics, indent=2))
for key, record in records.items():
    plt.semilogy(record["loss"], label=str(metrics[key]["visual_tokens"]) +
                 (" визуальных позиций" if DEMO_LANGUAGE == "ru" else " visual positions"))
plt.xlabel("Шаг оптимизации" if DEMO_LANGUAGE == "ru" else "Optimization step")
plt.ylabel("Обучающая кросс-энтропия" if DEMO_LANGUAGE == "ru" else "Training cross-entropy")
plt.legend(); plt.tight_layout(); plt.show()

На данном seed режим с 16 позициями правильно отвечает на 16 из 16 отложенных изображений, а с четырьмя — на 6. Пустые изображения дают 4 и 6 правильных ответов соответственно. Поэтому во втором режиме одной допустимой формы ответа недостаточно, чтобы подтвердить использование зрения. Конкатенация сама по себе не теряет пикселей; различаются параметры и ход обучения. Для вывода об архитектуре нужны повторные запуски и согласованные условия.

Второй опыт читает уже сохранённые ответы первой части. Допустимы только последовательности из одного ответа left/equal/right и EOS; всё остальное ведёт к уточнению. Разрешённое действие здесь лишь создаёт запись о метке на схеме. Отделим проверку формы ответа от проверки того, правильно ли модель поняла изображение.

In [ ]:
# Consume saved predictions; routing and scoring never invoke the VLM again.
ALLOWED = {1: "mark_left", 2: "mark_equal", 3: "mark_right"}

def route_saved(tokens):
    if (not isinstance(tokens, list) or len(tokens) != 2
            or any(type(t) is not int for t in tokens)
            or tokens[1] != 4 or tokens[0] not in ALLOWED):
        return "clarify"
    return ALLOWED[tokens[0]]

traces = {}
for key, record in records.items():
    log = []
    for request_id, (prediction, target) in enumerate(zip(record["predictions"], record["targets"])):
        action = route_saved(prediction)
        # The mock tool only records an annotation; it has no external side effect.
        assert action in set(ALLOWED.values()) | {"clarify"}
        log.append(dict(request_id=request_id, prediction=prediction, action=action,
                        correct=action == route_saved(target)))
    assert len(log) == metrics[key]["test_images"] == 16
    assert sum(row["correct"] for row in log) == metrics[key]["exact_correct"]
    traces[key] = log
assert route_saved(["delete_files", 4]) == "clarify"
assert route_saved([True, 4]) == "clarify"
assert route_saved([1, 0]) == "clarify"
assert json.loads(json.dumps(traces)) == traces
print({key: {"recorded_actions": len(rows), "correct": sum(r["correct"] for r in rows)}
       for key, rows in traces.items()})

Все записанные действия входят в разрешённый набор, но это не делает их правильными. Число правильных отметок совпадает с числом верных сохранённых ответов: маршрутизатор не исправляет ошибку восприятия. Проверки отклоняют неизвестное действие, булев ID и неверное завершение; JSON сохраняет связь между запросом, ответом и отметкой.

## Переход к полному проекту

В этом наборе отложена яркость, а не пары высот: не называйте результат переносом на новые композиции. Сравнение меняет также параметры сети, а число шагов одинаково. Для DocVQA зафиксируйте разбиение по документам, процессор каждой модели, бюджет визуальных токенов и реальное время стадий. Общий размер картинки не делает процессоры одинаковыми. Ошибка VLM может породить формально допустимое, но неверное действие: отдельно измеряйте корректность намерения и допустимость действия.

### Минимальный отчёт

1. Версии данных, модели, кода и seed.
2. Основная метрика, а также качество, стоимость, задержка и память там, где это применимо.
3. Базовая линия, одинаковый бюджет и хотя бы один отрицательный контроль.
4. Неудачные прогоны и границы обоснованных выводов.